In [ ]:
# ============================================================
# HANDWRITTEN NUMBER RECOGNITION USING CNN
# Dataset: MNIST
# Framework: TensorFlow / Keras
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

import seaborn as sns


print("TensorFlow Version:", tf.__version__)


# ============================================================
# 2. LOAD MNIST DATASET
# ============================================================

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)


# ============================================================
# 3. VISUALIZE SAMPLE IMAGES
# ============================================================

plt.figure(figsize=(10, 5))

for i in range(10):

    plt.subplot(2, 5, i + 1)

    plt.imshow(x_train[i], cmap="gray")

    plt.title(f"Label: {y_train[i]}")

    plt.axis("off")

plt.tight_layout()
plt.show()


# ============================================================
# 4. IMAGE PREPROCESSING
# ============================================================

# Normalize pixel values from [0,255] to [0,1]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0


# CNN expects:
# (number of images, height, width, channels)

x_train = x_train.reshape(
    x_train.shape[0],
    28,
    28,
    1
)

x_test = x_test.reshape(
    x_test.shape[0],
    28,
    28,
    1
)


print("After preprocessing:")
print("Training:", x_train.shape)
print("Testing:", x_test.shape)


# ============================================================
# 5. ONE-HOT ENCODE LABELS
# ============================================================

y_train = to_categorical(
    y_train,
    num_classes=10
)

y_test = to_categorical(
    y_test,
    num_classes=10
)


print("Encoded label shape:", y_train.shape)


# ============================================================
# 6. BUILD CNN MODEL
# ============================================================

model = Sequential([

    # Convolution Layer 1
    Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation="relu",
        input_shape=(28, 28, 1)
    ),

    # Pooling Layer 1
    MaxPooling2D(
        pool_size=(2, 2)
    ),

    # Convolution Layer 2
    Conv2D(
        filters=64,
        kernel_size=(3, 3),
        activation="relu"
    ),

    # Pooling Layer 2
    MaxPooling2D(
        pool_size=(2, 2)
    ),

    # Convert feature maps into vector
    Flatten(),

    # Fully Connected Layer
    Dense(
        128,
        activation="relu"
    ),

    # Dropout to reduce overfitting
    Dropout(0.5),

    # Output layer
    Dense(
        10,
        activation="softmax"
    )
])


# ============================================================
# 7. DISPLAY MODEL ARCHITECTURE
# ============================================================

model.summary()


# ============================================================
# 8. COMPILE MODEL
# ============================================================

model.compile(
    optimizer="adam",

    loss="categorical_crossentropy",

    metrics=["accuracy"]
)


# ============================================================
# 9. TRAIN CNN MODEL
# ============================================================

history = model.fit(

    x_train,
    y_train,

    validation_split=0.1,

    epochs=10,

    batch_size=128,

    verbose=1
)


# ============================================================
# 10. EVALUATE MODEL
# ============================================================

test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("\nTest Loss:", test_loss)

print("Test Accuracy:", test_accuracy)


# ============================================================
# 11. PLOT TRAINING / VALIDATION ACCURACY
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title("Training and Validation Accuracy")

plt.legend()

plt.grid()

plt.show()


# ============================================================
# 12. PLOT TRAINING / VALIDATION LOSS
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training and Validation Loss")

plt.legend()

plt.grid()

plt.show()


# ============================================================
# 13. MAKE PREDICTIONS
# ============================================================

predictions = model.predict(x_test)

predicted_labels = np.argmax(
    predictions,
    axis=1
)

true_labels = np.argmax(
    y_test,
    axis=1
)


# ============================================================
# 14. DISPLAY SAMPLE PREDICTIONS
# ============================================================

plt.figure(figsize=(12, 8))

for i in range(20):

    plt.subplot(4, 5, i + 1)

    plt.imshow(
        x_test[i].reshape(28, 28),
        cmap="gray"
    )

    plt.title(
        f"Actual: {true_labels[i]}\n"
        f"Predicted: {predicted_labels[i]}"
    )

    plt.axis("off")

plt.tight_layout()

plt.show()


# ============================================================
# 15. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    true_labels,
    predicted_labels
)

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=range(10),
    yticklabels=range(10)
)

plt.xlabel("Predicted Label")

plt.ylabel("True Label")

plt.title("MNIST CNN Confusion Matrix")

plt.show()


# ============================================================
# 16. CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        true_labels,
        predicted_labels
    )
)


# ============================================================
# 17. PREDICT A SINGLE IMAGE
# ============================================================

index = 100

image = x_test[index]

prediction = model.predict(
    image.reshape(1, 28, 28, 1)
)

predicted_digit = np.argmax(prediction)

confidence = np.max(prediction) * 100


plt.figure(figsize=(4, 4))

plt.imshow(
    image.reshape(28, 28),
    cmap="gray"
)

plt.title(
    f"Predicted Digit: {predicted_digit}\n"
    f"Confidence: {confidence:.2f}%"
)

plt.axis("off")

plt.show()


print("Actual Digit:", true_labels[index])

print("Predicted Digit:", predicted_digit)

print(f"Confidence: {confidence:.2f}%")


# ============================================================
# 18. SAVE MODEL
# ============================================================

model.save("mnist_cnn_model.keras")

print("\nModel saved as mnist_cnn_model.keras")